# G-I-A Framework: Conceptual Demonstration

This notebook provides an illustrative walkthrough of the **Grounding-Instructibility-Alignment (G-I-A)** framework introduced in our survey paper *"Neuro-Symbolic AI for Cybersecurity: State of the Art, Challenges, and Opportunities"*.

**Important:** G-I-A is presented as an *analytical framework* for examining NeSy cybersecurity systems, not as a computable scoring algorithm. The code below demonstrates the *conceptual structure* of each dimension.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 12,
    'figure.dpi': 150
})

## 1. G-I-A Formal Definitions

The three G-I-A dimensions are defined as follows:

**Grounding Quality** measures how well the system connects outputs to cybersecurity concepts:

$$\mathcal{G}(\theta, \mathcal{K}) = \frac{1}{|\mathcal{Z}|} \sum_{c \in \mathcal{Z}} \text{Consistency}(\Phi_\theta(x_c), \Psi_{\mathcal{K}}(x_c, c))$$

**Instructibility** quantifies responsiveness to analyst feedback:

$$\mathcal{I}(\theta, \mathcal{K}, \mathcal{H}) = \mathbb{E}_{h \in \mathcal{H}} \left[ \text{Adaptation}(\Delta\theta_h, \Delta\mathcal{K}_h) \right]$$

**Alignment** ensures consistency with organizational objectives:

$$\mathcal{A}(\theta, \mathcal{K}, \mathcal{O}) = \sum_{o \in \mathcal{O}} w_o \cdot \text{Objective}(\Phi_\theta, \Psi_{\mathcal{K}}, o)$$

In [ ]:
class GIAFramework:
    """Conceptual implementation of the G-I-A analytical framework.
    
    This is an illustrative demonstration, not a production scoring tool.
    The Consistency, Adaptation, and Objective functions are intentionally
    left abstract in the paper; here we provide simple instantiations
    for demonstration purposes only.
    """
    
    def __init__(self, cybersecurity_concepts, objectives, objective_weights=None):
        self.concepts = cybersecurity_concepts
        self.objectives = objectives
        if objective_weights is None:
            self.weights = {o: 1.0/len(objectives) for o in objectives}
        else:
            self.weights = objective_weights
    
    def grounding_quality(self, neural_predictions, symbolic_reasoning):
        """Measure consistency between neural outputs and symbolic reasoning
        across cybersecurity concepts.
        
        Args:
            neural_predictions: dict mapping concept -> neural output (0-1)
            symbolic_reasoning: dict mapping concept -> symbolic output (0-1)
        """
        consistencies = []
        for concept in self.concepts:
            if concept in neural_predictions and concept in symbolic_reasoning:
                # Simple consistency: 1 - |neural - symbolic|
                consistency = 1.0 - abs(
                    neural_predictions[concept] - symbolic_reasoning[concept]
                )
                consistencies.append(consistency)
        return np.mean(consistencies) if consistencies else 0.0
    
    def instructibility(self, pre_feedback_perf, post_feedback_perf):
        """Measure how effectively the system adapts to analyst feedback.
        
        Args:
            pre_feedback_perf: performance before feedback (0-1)
            post_feedback_perf: performance after feedback (0-1)
        """
        adaptation = max(0, post_feedback_perf - pre_feedback_perf)
        # Normalize to 0-1 scale
        return min(adaptation / 0.5, 1.0)  # 50% improvement = perfect score
    
    def alignment(self, objective_scores):
        """Measure weighted alignment with organizational objectives.
        
        Args:
            objective_scores: dict mapping objective -> score (0-1)
        """
        total = 0.0
        for obj in self.objectives:
            if obj in objective_scores:
                total += self.weights[obj] * objective_scores[obj]
        return total
    
    def integrated_loss(self, L_N, G, I, A, lambda_G=1.0, lambda_I=1.0, lambda_A=1.0):
        """Compute integrated G-I-A optimization objective.
        
        L_GIA = L_N - lambda_G * G - lambda_I * I - lambda_A * A
        """
        return L_N - lambda_G * G - lambda_I * I - lambda_A * A

## 2. Illustrative Example: Scoring a Hypothetical NeSy IDS

We walk through scoring a hypothetical Knowledge-Graph-enhanced Intrusion Detection System.

In [ ]:
# Define cybersecurity concepts and objectives
concepts = [
    'lateral_movement', 'privilege_escalation', 'data_exfiltration',
    'command_and_control', 'initial_access', 'persistence'
]

objectives = [
    'minimize_false_positives', 'maximize_detection_rate',
    'provide_explanations', 'comply_with_policy'
]

objective_weights = {
    'minimize_false_positives': 0.3,
    'maximize_detection_rate': 0.3,
    'provide_explanations': 0.2,
    'comply_with_policy': 0.2
}

gia = GIAFramework(concepts, objectives, objective_weights)

# --- Grounding: How well do neural and symbolic components agree? ---
neural_preds = {
    'lateral_movement': 0.85, 'privilege_escalation': 0.72,
    'data_exfiltration': 0.91, 'command_and_control': 0.68,
    'initial_access': 0.79, 'persistence': 0.83
}
symbolic_preds = {
    'lateral_movement': 0.90, 'privilege_escalation': 0.80,
    'data_exfiltration': 0.88, 'command_and_control': 0.75,
    'initial_access': 0.82, 'persistence': 0.78
}

G = gia.grounding_quality(neural_preds, symbolic_preds)

# --- Instructibility: How much does the system improve after analyst feedback? ---
I = gia.instructibility(pre_feedback_perf=0.78, post_feedback_perf=0.91)

# --- Alignment: How well does the system serve organizational objectives? ---
obj_scores = {
    'minimize_false_positives': 0.85,
    'maximize_detection_rate': 0.92,
    'provide_explanations': 0.78,
    'comply_with_policy': 0.90
}
A = gia.alignment(obj_scores)

print("=" * 55)
print("G-I-A Assessment: Hypothetical KG-Enhanced NeSy IDS")
print("=" * 55)
print(f"  Grounding Quality (G):    {G:.3f}  (scale: 0-1)")
print(f"  Instructibility (I):      {I:.3f}  (scale: 0-1)")
print(f"  Alignment (A):            {A:.3f}  (scale: 0-1)")
print(f"\n  Scaled to /5 (as in Table 2):")
print(f"  G = {G*5:.1f}/5,  I = {I*5:.1f}/5,  A = {A*5:.1f}/5")

# Integrated loss
L_N = 0.35  # hypothetical neural training loss
L_GIA = gia.integrated_loss(L_N, G, I, A, lambda_G=1.0, lambda_I=0.8, lambda_A=1.2)
print(f"\n  Integrated G-I-A Loss:    {L_GIA:.3f}")
print(f"  (lower is better; negative means G-I-A terms dominate)")

## 3. Comparing Systems from the Survey

We load the G-I-A scores from Table 2 and visualize them as radar charts.

In [ ]:
# Load G-I-A scores from Table 2
systems = {
    'KnowGraph': {'G': 4.2, 'I': 3.1, 'A': 3.8, 'tier': 'A'},
    'HPTSA': {'G': 3.5, 'I': 2.8, 'A': 2.1, 'tier': 'C'},
    'VulnBot': {'G': 3.8, 'I': 3.5, 'A': 4.1, 'tier': 'B'},
    'H-MARL Defense': {'G': 3.7, 'I': 3.8, 'A': 4.0, 'tier': 'C'},
    'LTN-IDS': {'G': 4.0, 'I': 3.4, 'A': 3.9, 'tier': 'A'},
    'IoT NeSy': {'G': 4.1, 'I': 3.6, 'A': 4.2, 'tier': 'A'},
}

# Radar chart
categories = ['Grounding', 'Instructibility', 'Alignment']
N = len(categories)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw=dict(polar=True))

# Color by tier
tier_colors = {'A': '#2E8B57', 'B': '#1F77B4', 'C': '#DC143C'}
tier_labels = {'A': 'Deep NeSy', 'B': 'Structured NeSy', 'C': 'Baseline'}

# Left: All systems overlaid
ax = axes[0]
for name, scores in systems.items():
    values = [scores['G'], scores['I'], scores['A']]
    values += values[:1]
    color = tier_colors[scores['tier']]
    ax.plot(angles, values, 'o-', linewidth=2, label=f"{name} ({tier_labels[scores['tier']]})",
            color=color, alpha=0.7)
    ax.fill(angles, values, alpha=0.05, color=color)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories, fontsize=13)
ax.set_ylim(0, 5)
ax.set_yticks([1, 2, 3, 4, 5])
ax.set_title('G-I-A Profiles: All Systems', fontsize=14, fontweight='bold', pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.4, 1.1), fontsize=10)

# Right: Type A vs Type C comparison
ax2 = axes[1]
type_a_avg = [np.mean([s['G'] for s in systems.values() if s['tier']=='A']),
              np.mean([s['I'] for s in systems.values() if s['tier']=='A']),
              np.mean([s['A'] for s in systems.values() if s['tier']=='A'])]
type_c_avg = [np.mean([s['G'] for s in systems.values() if s['tier']=='C']),
              np.mean([s['I'] for s in systems.values() if s['tier']=='C']),
              np.mean([s['A'] for s in systems.values() if s['tier']=='C'])]

type_a_avg += type_a_avg[:1]
type_c_avg += type_c_avg[:1]

ax2.plot(angles, type_a_avg, 'o-', linewidth=2.5, label='Type A (Deep NeSy)', color='#2E8B57')
ax2.fill(angles, type_a_avg, alpha=0.15, color='#2E8B57')
ax2.plot(angles, type_c_avg, 's--', linewidth=2.5, label='Type C (Baselines)', color='#DC143C')
ax2.fill(angles, type_c_avg, alpha=0.15, color='#DC143C')

ax2.set_xticks(angles[:-1])
ax2.set_xticklabels(categories, fontsize=13)
ax2.set_ylim(0, 5)
ax2.set_yticks([1, 2, 3, 4, 5])
ax2.set_title('Average G-I-A: Deep NeSy vs Baselines', fontsize=14, fontweight='bold', pad=20)
ax2.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11)

plt.tight_layout()
plt.savefig('../figures/gia_radar_comparison.pdf', bbox_inches='tight')
plt.savefig('../figures/gia_radar_comparison.png', bbox_inches='tight', dpi=300)
plt.show()

print("\nKey observation: Type A (Deep NeSy) systems show consistently")
print("higher Grounding and Alignment scores, while the gap in")
print("Instructibility is smaller -- suggesting this dimension")
print("remains a shared challenge across integration depths.")

## 4. Survey Corpus Overview

Quick summary statistics from the 103-paper catalog.

In [ ]:
# Load catalog
df = pd.read_csv('../data/paper_catalog.csv')

print(f"Total papers: {len(df)}")
print(f"\nBy Integration Tier:")
tier_counts = df['tier'].value_counts().sort_index()
tier_names = {'A': 'Deep NeSy', 'B': 'Structured NeSy', 'C': 'Contextual Baselines'}
for tier, count in tier_counts.items():
    pct = count / len(df) * 100
    print(f"  Type {tier} ({tier_names[tier]}): {count} papers ({pct:.1f}%)")

print(f"\nBy Subtype (Type B breakdown):")
type_b = df[df['tier'] == 'B']
for subtype, count in type_b['subtype'].value_counts().items():
    print(f"  {subtype}: {count} papers")

print(f"\nBy Year:")
for year, count in df['year'].value_counts().sort_index().items():
    print(f"  {year}: {count} papers")

In [ ]:
# Visualization: Tier distribution and domain coverage
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Tier distribution
colors_tier = ['#2E8B57', '#1F77B4', '#808080']
labels_tier = [f'Type A\nDeep NeSy\n({tier_counts["A"]} papers)',
               f'Type B\nStructured NeSy\n({tier_counts["B"]} papers)',
               f'Type C\nBaselines\n({tier_counts["C"]} papers)']
axes[0].pie(tier_counts.values, labels=labels_tier, colors=colors_tier,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 12})
axes[0].set_title('Integration Tier Distribution', fontsize=14, fontweight='bold')

# Right: Top domains
domain_counts = df['domain'].value_counts().head(10)
axes[1].barh(range(len(domain_counts)), domain_counts.values, color='#1F77B4', alpha=0.8)
axes[1].set_yticks(range(len(domain_counts)))
axes[1].set_yticklabels(domain_counts.index, fontsize=11)
axes[1].set_xlabel('Number of Papers', fontsize=13)
axes[1].set_title('Top Application Domains', fontsize=14, fontweight='bold')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../figures/corpus_overview.pdf', bbox_inches='tight')
plt.savefig('../figures/corpus_overview.png', bbox_inches='tight', dpi=300)
plt.show()

## Note on Scope

This notebook is a **conceptual demonstration** of the G-I-A framework. The `Consistency`, `Adaptation`, and `Objective` functions used here are simple illustrative instantiations. In the paper, these are deliberately left abstract because:

1. Different cybersecurity domains require different operationalizations
2. The framework's value lies in structuring analysis, not prescribing specific metrics
3. Concrete instantiation and validation is identified as a key future research direction

For the full discussion, see Section 2.1 of the paper.